# 🚦 RoadSense — Full Training Pipeline (Google Colab GPU)
**Group 4 | Atria University | Neural Network & CNN Sprint**

This notebook:
1. Clones the RoadSense repo from GitHub
2. Installs dependencies
3. Downloads the GTSRB dataset (~300MB)
4. Trains all 4 models (Baseline CNN, Improved CNN, ResNet-18, MobileNetV2)
5. Runs evaluation + robustness benchmarks
6. Generates comparison plots
7. Saves all results to Google Drive

⏱️ **Estimated total time on T4 GPU: ~35–50 minutes**

---
### ⚠️ BEFORE RUNNING:
- Go to **Runtime → Change runtime type → T4 GPU** ✅
- Make sure you are signed in to Google


In [ ]:
# ── STEP 0: Check GPU ──────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
print('Device name  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('PyTorch      :', torch.__version__)
!nvidia-smi | head -20

In [ ]:
# ── STEP 1: Mount Google Drive (to save checkpoints permanently) ─
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/RoadSense_Results'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'Results will be saved to: {DRIVE_SAVE_DIR}')

In [ ]:
# ── STEP 2: Clone the repo ─────────────────────────────────────
# ⚠️ CHANGE THIS to your actual GitHub repo URL!
GITHUB_URL = 'https://github.com/rajukumar-tech/RoadSense-Robust-Traffic-Sign-Recognition'

import os
if os.path.exists('/content/ROADSENSE'):
    print('Repo already cloned, pulling latest...')
    !cd /content/ROADSENSE && git pull
else:
    !git clone {GITHUB_URL} /content/ROADSENSE

os.chdir('/content/ROADSENSE')
!ls -la

In [ ]:
# ── STEP 3: Install dependencies ──────────────────────────────
!pip install -q opencv-python-headless tqdm seaborn scikit-learn matplotlib pandas numpy requests pyarrow
print('All dependencies installed!')

In [ ]:
# ── STEP 4: Download GTSRB dataset ────────────────────────────
# Downloads ~300MB from HuggingFace, extracts 51,000 images
# Takes ~3-5 minutes depending on Colab server speed
import time
t = time.time()
!python src/download_data.py
print(f'\nDataset ready in {(time.time()-t)/60:.1f} minutes')

In [ ]:
# ── STEP 5: Verify dataset ─────────────────────────────────────
import os
train_count = sum(len(f) for _, _, f in os.walk('data/Train'))
test_count  = len([f for f in os.listdir('data/Test') if f.endswith('.png')])
print(f'Train images: {train_count:,}  (expected ~39,209)')
print(f'Test images : {test_count:,}  (expected ~12,630)')

# Verify all src modules load
import sys
sys.path.insert(0, '/content/ROADSENSE')
import src.models, src.train, src.evaluate, src.robustness_bench
print('All modules imported successfully!')

In [ ]:
# ── STEP 6: TRAIN ALL 4 MODELS ────────────────────────────────
# This is the main training cell.
# On T4 GPU: ~35-50 minutes total
# On A100 GPU: ~10-15 minutes total

import time
t_total = time.time()

!python run_all_phases.py

elapsed = (time.time() - t_total) / 60
print(f'\n✅ COMPLETE! Total time: {elapsed:.1f} minutes')

In [ ]:
# ── STEP 7: Verify all output files exist ─────────────────────
from pathlib import Path

expected = [
    'checkpoints/best_baseline_cnn.pth',
    'checkpoints/best_improved_cnn.pth',
    'checkpoints/best_resnet18.pth',
    'checkpoints/best_mobilenet_v2.pth',
    'results/model_comparison.csv',
    'results/baseline_cnn_robustness.csv',
    'results/improved_cnn_robustness.csv',
    'results/resnet18_robustness.csv',
    'results/mobilenet_v2_robustness.csv',
    'results/figures/four_way_model_comparison.png',
    'results/figures/phase4a_augmentation_comparison.png',
    'results/figures/all_models_robustness_overlay.png',
    'results/figures/phase4_learning_curves_comparison.png',
]

all_ok = True
for f in expected:
    p = Path(f)
    status = '✅' if p.exists() else '❌ MISSING'
    size = f'({p.stat().st_size/1e6:.1f} MB)' if p.exists() else ''
    print(f'{status}  {f}  {size}')
    if not p.exists():
        all_ok = False

print('\n' + ('🎉 ALL FILES PRESENT!' if all_ok else '⚠️ Some files missing — check logs above'))

In [ ]:
# ── STEP 8: Show comparison table ─────────────────────────────
import pandas as pd
df = pd.read_csv('results/model_comparison.csv')
print('\n📊 FINAL MODEL COMPARISON')
print('=' * 65)
print(df.to_string(index=False))

In [ ]:
# ── STEP 9: Display all comparison figures ────────────────────
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

figures = [
    ('results/figures/four_way_model_comparison.png',      'Four-Way Model Comparison'),
    ('results/figures/phase4a_augmentation_comparison.png','Phase 4A: Augmentation Effect'),
    ('results/figures/all_models_robustness_overlay.png',  'Robustness Degradation Curves'),
    ('results/figures/phase4_learning_curves_comparison.png','Learning Curves — All Models'),
]

for path, title in figures:
    if Path(path).exists():
        print(f'\n📈 {title}')
        display(Image(path, width=900))

In [ ]:
# ── STEP 10: Save everything to Google Drive ──────────────────
import shutil, os

DRIVE_SAVE_DIR = '/content/drive/MyDrive/RoadSense_Results'

# Copy checkpoints
ckpt_dst = os.path.join(DRIVE_SAVE_DIR, 'checkpoints')
os.makedirs(ckpt_dst, exist_ok=True)
for f in Path('checkpoints').glob('*.pth'):
    shutil.copy(f, ckpt_dst)
    print(f'Saved checkpoint: {f.name}')

# Copy results
res_dst = os.path.join(DRIVE_SAVE_DIR, 'results')
if os.path.exists(res_dst):
    shutil.rmtree(res_dst)
shutil.copytree('results', res_dst)
print(f'Saved all results to Google Drive!')

print(f'\n✅ Everything saved to: {DRIVE_SAVE_DIR}')
print('You can now download from Google Drive!')

In [ ]:
# ── STEP 11: Download a ZIP of all results directly ───────────
# Alternative: download without using Drive
!zip -r /content/RoadSense_outputs.zip checkpoints/ results/
from google.colab import files
files.download('/content/RoadSense_outputs.zip')
print('ZIP download triggered!')

---
## 🎉 Done!

All files are saved to:
- **Google Drive** → `My Drive/RoadSense_Results/`
- **ZIP download** → `RoadSense_outputs.zip` (auto-downloaded)

### Next Steps
1. Copy `.pth` files to your local `checkpoints/` folder
2. Copy `results/` folder back to your local machine
3. Open `notebooks/04_evaluation_and_explainability.ipynb` for Phase 5 (Grad-CAM)

### Files to copy back locally:
```
checkpoints/best_baseline_cnn.pth
checkpoints/best_improved_cnn.pth  
checkpoints/best_resnet18.pth
checkpoints/best_mobilenet_v2.pth
results/model_comparison.csv
results/figures/*.png
results/*_robustness.csv
```
